# Instalando os pacotes

In [ ]:
import pandas as pd
import numpy as np
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager
import zipfile

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


# Configurando o navegador

In [2]:
# Configura a pasta onde o arquivo será salvo (na mesma pasta do seu notebook)
download_dir = os.path.join(os.getcwd(), "downloads_bcb")
os.makedirs(download_dir, exist_ok=True)

# Configurações do Chrome para evitar a janela de "Salvar Como"
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "directory_upgrade": True
}
options.add_experimental_option("prefs", prefs)

# Inicializa o navegador
print("Abrindo o navegador...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

Abrindo o navegador...


# Processo de Web Scraping

In [3]:
print("Acessando o site do BCB para baixar a base histórica completa...")
driver.get("https://www.bcb.gov.br/estabilidadefinanceira/consorciobd")

try:
    print("Iniciando varredura dinâmica em busca do menu ...")
    
    menu_encontrado = False
    select_element = None
    
    # Loop de tentativas: Vasculha os iframes por até 30 segundos
    tempo_limite = time.time() + 30
    
    while time.time() < tempo_limite:
        driver.switch_to.default_content() # Volta para a página principal
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        
        for iframe in iframes:
            try:
                driver.switch_to.frame(iframe) # Entra no iframe
                elementos = driver.find_elements(By.ID, "Consorcios")
                if len(elementos) > 0:
                    select_element = elementos[0]
                    menu_encontrado = True
                    break # Achou o menu
            except:
                pass # Ignora erros de iframes inacessíveis
                
        if menu_encontrado:
            break 
            
        time.sleep(2) # Pausa antes de tentar de novo
        
    if not menu_encontrado:
        raise Exception("Tempo esgotado (30s). O site do Banco Central está muito lento ou fora do ar hoje.")

    print("Robô encontrou o menu com sucesso!")
    menu = Select(select_element)
    
    # Lista para guardar todos os links
    arquivos_para_baixar = []
    print("Mapeando TODOS os arquivos (do mais recente até Janeiro/1997)...")
    
    # Percorre o menu do topo até a base
    for opcao in menu.options:
        nome_mes = opcao.text
        caminho_arquivo = opcao.get_attribute("value")
        
        # Prevenção: só guarda se realmente for um link de arquivo ZIP
        if not caminho_arquivo or ".zip" not in caminho_arquivo.lower():
            continue
            
        # Monta o link completo
        link_completo = f"https://www.bcb.gov.br{caminho_arquivo}"
        arquivos_para_baixar.append((nome_mes, link_completo))
        
        # Condição de parada final no fundo do poço histórico
        if "Janeiro/1997" in nome_mes:
            print("Limite histórico (Janeiro/1997) encontrado. Finalizando o mapeamento.")
            break
            
    print(f"\nTotal histórico de arquivos mapeados: {len(arquivos_para_baixar)}")
    print("Atenção: Como a lista é grande, este processo vai levar alguns minutos para não sobrecarregar o site.")
    print("Iniciando a fila de downloads...\n")
    
    # Executa os downloads
    for i, (mes, link) in enumerate(arquivos_para_baixar, 1):
        mes_limpo = mes.split('(')[0].strip() 
        print(f"[{i}/{len(arquivos_para_baixar)}] Baixando: {mes_limpo}...")
        
        driver.get(link)
        
        # Pausa de 3 segundos rigorosa (evita que o BCB bloqueie seu IP por excesso de requisições)
        time.sleep(3) 
        
    print("\nTodos os comandos de download foram enviados para o Chrome!")
    print("Aguardando 20 segundos extras para garantir que os últimos downloads terminem em segundo plano...")
    time.sleep(20)
    
    print(f"A base completa ({len(arquivos_para_baixar)} arquivos) está salva na pasta: {download_dir}")

except Exception as e:
    print(f"Ocorreu um erro durante a execução: {e}")

Acessando o site do BCB para baixar a base histórica completa...
Iniciando varredura dinâmica em busca do menu ...
Robô encontrou o menu com sucesso!
Mapeando TODOS os arquivos (do mais recente até Janeiro/1997)...
Limite histórico (Janeiro/1997) encontrado. Finalizando o mapeamento.

Total histórico de arquivos mapeados: 351
⏳ Atenção: Como a lista é grande, este processo vai levar alguns minutos para não sobrecarregar o site.
Iniciando a fila de downloads...

[1/351] Baixando: Março/2026...
[2/351] Baixando: Fevereiro/2026...
[3/351] Baixando: Janeiro/2026...
[4/351] Baixando: Dezembro/2025...
[5/351] Baixando: Novembro/2025...
[6/351] Baixando: Outubro/2025...
[7/351] Baixando: Setembro/2025...
[8/351] Baixando: Agosto/2025...
[9/351] Baixando: Julho/2025...
[10/351] Baixando: Junho/2025...
[11/351] Baixando: Maio/2025...
[12/351] Baixando: Abril/2025...
[13/351] Baixando: Março/2025...
[14/351] Baixando: Fevereiro/2025...
[15/351] Baixando: Janeiro/2025...
[16/351] Baixando: Dezemb

In [4]:
driver.quit()
print("Navegador fechado e recursos liberados.")

Navegador fechado e recursos liberados.


# Descompactar os arquivos zipados

In [5]:
#Define os caminhos das pastas
pasta_downloads = os.path.join(os.getcwd(), "downloads_bcb")
pasta_destino = os.path.join(os.getcwd(), "dados_extraidos")

# Cria a pasta de destino caso ela não exista
os.makedirs(pasta_destino, exist_ok=True)

#Lista todos os arquivos da pasta de downloads
arquivos = os.listdir(pasta_downloads)
arquivos_zip = [f for f in arquivos if f.lower().endswith('.zip')]

print(f"Encontrados {len(arquivos_zip)} arquivos ZIP para descompactar.")
print("Iniciando a extração...\n")

# Contador de sucesso
sucesso = 0

#Percorre e extrai cada arquivo ZIP
for i, arquivo in enumerate(arquivos_zip, 1):
    caminho_zip = os.path.join(pasta_downloads, arquivo)
    
    # Cria uma subpasta para cada mês para manter os dados organizados (opcional)
    # Exemplo: "202403Consorcios_UF.zip" vira uma pasta com esse mesmo nome
    nome_subpasta = arquivo.replace('.zip', '').replace('.ZIP', '')
    caminho_subpasta_destino = os.path.join(pasta_destino, nome_subpasta)
    os.makedirs(caminho_subpasta_destino, exist_ok=True)
    
    try:
        # Abre e extrai o arquivo ZIP
        with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
            zip_ref.extractall(caminho_subpasta_destino)
        
        print(f"[{i}/{len(arquivos_zip)}] Extraído com sucesso: {arquivo}")
        sucesso += 1
        
    except zipfile.BadZipFile:
        print(f"Erro [{i}/{len(arquivos_zip)}]: O arquivo {arquivo} está corrompido (download incompleto).")
    except Exception as e:
        print(f"Erro [{i}/{len(arquivos_zip)}] ao extrair {arquivo}: {e}")

print(f"\n Processo concluído! {sucesso} de {len(arquivos_zip)} arquivos foram extraídos.")
print(f"Os dados finais estão organizados na pasta: {pasta_destino}")

Encontrados 351 arquivos ZIP para descompactar.
Iniciando a extração...

[1/351] Extraído com sucesso: 199701Consorcios.zip
[2/351] Extraído com sucesso: 199702Consorcios.zip
[3/351] Extraído com sucesso: 199703Consorcios.zip
[4/351] Extraído com sucesso: 199704Consorcios.zip
[5/351] Extraído com sucesso: 199705Consorcios.zip
[6/351] Extraído com sucesso: 199706Consorcios.zip
[7/351] Extraído com sucesso: 199707Consorcios.zip
[8/351] Extraído com sucesso: 199708Consorcios.zip
[9/351] Extraído com sucesso: 199709Consorcios.zip
[10/351] Extraído com sucesso: 199710Consorcios.zip
[11/351] Extraído com sucesso: 199711Consorcios.zip
[12/351] Extraído com sucesso: 199712Consorcios.zip
[13/351] Extraído com sucesso: 199801Consorcios.zip
[14/351] Extraído com sucesso: 199802Consorcios.zip
[15/351] Extraído com sucesso: 199803Consorcios.zip
[16/351] Extraído com sucesso: 199804Consorcios.zip
[17/351] Extraído com sucesso: 199805Consorcios.zip
[18/351] Extraído com sucesso: 199806Consorcios.zip


# Leitura dos arquivos de Imoves e agregação em um único Dataframe

In [7]:
# Define os caminhos
pasta_base = os.path.join(os.getcwd(), "dados_extraidos")
termo_busca = "Bens_Imoveis_Grupos"
caminho_salvamento = os.path.join(os.getcwd(), "Bens_Imoveis_Consolidado.csv")

lista_dataframes = []
arquivos_processados = 0

print(f"Iniciando a busca por arquivos contendo '{termo_busca}'...")

# Percorre todas as pastas e subpastas dentro de 'dados_extraidos'
for pasta_raiz, _, arquivos in os.walk(pasta_base):
    for arquivo in arquivos:
        
        # Verifica se o nome do arquivo contém o termo desejado e é um arquivo de texto/csv
        if termo_busca in arquivo and (arquivo.endswith('.csv') or arquivo.endswith('.txt')):
            caminho_completo = os.path.join(pasta_raiz, arquivo)
            
            try:
                # Lê o arquivo
                df_temporario = pd.read_csv(caminho_completo, sep=';', encoding='latin1', low_memory=False)
                
                # Opcional: Cria uma coluna indicando de qual arquivo esses dados vieram
                df_temporario['Arquivo_Origem'] = arquivo
                
                lista_dataframes.append(df_temporario)
                arquivos_processados += 1
                
                print(f"Lido: {arquivo} | Linhas: {len(df_temporario)}")
                
            except Exception as e:
                print(f"Erro ao ler o arquivo {arquivo}: {e}")

# Junta tudo e salva
if len(lista_dataframes) > 0:
    print("\nEmpilhando todos os arquivos. Isso pode levar alguns segundos...")
    
    # Concatena todos os DataFrames da lista em um único chamado df_imoveis
    df_imoveis_consolidado = pd.concat(lista_dataframes, ignore_index=True)
    
    print(f"O DataFrame 'df_imoveis_consolidado' possui {df_imoveis_consolidado.shape[0]} linhas e {df_imoveis_consolidado.shape[1]} colunas.")
    
    # Salva o arquivo final no seu computador
    df_imoveis_consolidado.to_csv(caminho_salvamento, sep=';', encoding='latin1', index=False)
    print(f"Arquivo final salvo em: {caminho_salvamento}")
    
    # Mostra as primeiras 5 linhas para você conferir os dados
    display(df_imoveis_consolidado.head())
    
else:
    print(f"Nenhum arquivo contendo '{termo_busca}' foi encontrado nas subpastas.")

Iniciando a busca por arquivos contendo 'Bens_Imoveis_Grupos'...
Lido: 199701Bens_Imoveis_Grupos.csv | Linhas: 240
Lido: 199702Bens_Imoveis_Grupos.csv | Linhas: 247
Lido: 199703Bens_Imoveis_Grupos.csv | Linhas: 250
Lido: 199704Bens_Imoveis_Grupos.csv | Linhas: 254
Lido: 199705Bens_Imoveis_Grupos.csv | Linhas: 261
Lido: 199706Bens_Imoveis_Grupos.csv | Linhas: 262
Lido: 199707Bens_Imoveis_Grupos.csv | Linhas: 266
Lido: 199708Bens_Imoveis_Grupos.csv | Linhas: 276
Lido: 199709Bens_Imoveis_Grupos.csv | Linhas: 279
Lido: 199710Bens_Imoveis_Grupos.csv | Linhas: 282
Lido: 199711Bens_Imoveis_Grupos.csv | Linhas: 290
Lido: 199712Bens_Imoveis_Grupos.csv | Linhas: 297
Lido: 199801Bens_Imoveis_Grupos.csv | Linhas: 300
Lido: 199802Bens_Imoveis_Grupos.csv | Linhas: 305
Lido: 199803Bens_Imoveis_Grupos.csv | Linhas: 310
Lido: 199804Bens_Imoveis_Grupos.csv | Linhas: 313
Lido: 199805Bens_Imoveis_Grupos.csv | Linhas: 300
Lido: 199806Bens_Imoveis_Grupos.csv | Linhas: 302
Lido: 199807Bens_Imoveis_Grupos.csv

,#Nome_da_Administradora,CNPJ_da_Administradora,Data_base,Código_do_grupo,Código_do_segmento,Número_da_assembléia_geral_ordinária,Valor_médio_do_bem,Índice_de_correção,Taxa_de_administração,Prazo_do_grupo_em_meses,Quantidade_de_cotas_ativas_em_dia,Quantidade_de_cotas_ativas_contempladas_inadimplentes,Quantidade_de_cotas_ativas_não_contempladas_inadimplentes,Quantidade_de_cotas_ativas_contempladas_no_mês,Quantidade_de_cotas_excluídas,Quantidade_de_cotas_ativas_quitadas,Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização,Arquivo_Origem
0,COPLAVEN - IMPEDIDA CRIAR NOVOS GRUPOS,1471788,199701,00C01,1,80,"46759,00",3,"11,00000",100,186,9,1,2,561,26,7,199701Bens_Imoveis_Grupos.csv
1,COPLAVEN - IMPEDIDA CRIAR NOVOS GRUPOS,1471788,199701,00C07,1,38,"9923,00",3,"11,00000",100,132,4,5,1,837,1,4,199701Bens_Imoveis_Grupos.csv
2,CCA ADM CONSORCIO LTDA - IMPEDIDA CRIAR NOVOS GRUPOS,2790467,199701,01I01,1,21,"35624,00",1,"9,90000",80,107,10,36,1,302,2,0,199701Bens_Imoveis_Grupos.csv
3,CCA ADM CONSORCIO LTDA - IMPEDIDA CRIAR NOVOS GRUPOS,2790467,199701,01I02,1,13,"38045,00",1,"9,50000",100,145,4,46,2,364,2,0,199701Bens_Imoveis_Grupos.csv
4,ACAUA ADM.CONS. - IMPEDIDA CRIAR NOVOS GRUPOS,15939432,199701,51,1,48,"33720,00",3,"12,00000",100,20,15,4,0,166,3,0,199701Bens_Imoveis_Grupos.csv
